In [76]:
import os,json,hashlib
import numpy as np
import sklearn,joblib
import sys
PROJECT_ROOT = "/Users/vidhimishra/Desktop/Anomaly Detection"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: /Users/vidhimishra/Desktop/Anomaly Detection


In [77]:
from src.models.isolation_forest import MultiChannelIsolationForest
from src.inference.live_window import preprocess_live_window
from src.utils.config import WINDOW_SIZE, STRIDE

MODEL_PATH = "models_saved/isoforest_all_channels_with_scalers.joblib"
TEST_DIR = "archive/data/data/test"
FIXTURES_PATH = "tests/fixtures/regression_fixtures.json"

print(f"scikit-learn {sklearn.__version__} | numpy {np.__version__} | joblib {joblib.__version__}")


scikit-learn 1.6.1 | numpy 1.26.4 | joblib 1.4.2


In [78]:
with open(FIXTURES_PATH) as f:
    fixtures = json.load(f)

meta = fixtures["meta"]
cases = fixtures["cases"]

actual_sha256 = hashlib.sha256(open(MODEL_PATH, "rb").read()).hexdigest()
sha_match = actual_sha256 == meta["model_sha256"]

print(f"Fixtures generated: {meta['generated_on']}")
print(f"Fixtures generated with: {meta['generated_with']}")
print(f"Current environment:     "
      f"{{'python': '{sys.version.split()[0]}', 'numpy': '{np.__version__}', "
      f"'sklearn': '{sklearn.__version__}', 'joblib': '{joblib.__version__}'}}")
print()
print(f"Model artifact sha256 matches fixtures' recorded hash: {sha_match}")
if not sha_match:
    print("  WARNING: the model artifact has changed since these fixtures were generated.")
    print("  A model change is expected to shift predictions -- if so, regenerate fixtures")
    print("  (see the 'Regenerating fixtures' cell at the bottom) rather than treating this")
    print("  run as a pass/fail signal.")

assert os.path.exists(MODEL_PATH), f"Model artifact not found at {MODEL_PATH}"

Fixtures generated: 2026-07-21
Fixtures generated with: {'python': '3.12.3', 'numpy': '2.4.4', 'sklearn': '1.8.0', 'joblib': '1.5.3'}
Current environment:     {'python': '3.12.4', 'numpy': '1.26.4', 'sklearn': '1.6.1', 'joblib': '1.4.2'}

Model artifact sha256 matches fixtures' recorded hash: True


In [79]:
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    manager = MultiChannelIsolationForest.load(MODEL_PATH)
    version_warnings = [str(w.message) for w in caught
                         if "InconsistentVersionWarning" in str(w.category.__name__)]

print(f"Loaded {len(manager.detectors)} channel detectors, {len(manager.scalers)} scalers.")
if version_warnings:
    print(f"\n{len(version_warnings)} sklearn InconsistentVersionWarning(s) raised on load "
          f"(artifact was pickled under an older scikit-learn -- see Known Limitations).")


Loaded 81 channel detectors, 81 scalers.


In [80]:
def rebuild_raw_window(case):
    if case["offset"] is not None:
        test_arr = np.load(os.path.join(TEST_DIR, f"{case['ch_id']}.npy"))
        off = case["offset"]
        raw_window = test_arr[off:off + WINDOW_SIZE]
    else:
        n_features = np.array(case["window_shape"])[1]
        name = case["synthetic"]
        if name == "zeros":
            raw_window = np.zeros((WINDOW_SIZE, n_features))
        elif name == "ones":
            raw_window = np.ones((WINDOW_SIZE, n_features))
        elif name == "constant_5":
            raw_window = np.full((WINDOW_SIZE, n_features), 5.0)
        else:
            raise ValueError(f"Unknown synthetic case: {name}")
    return raw_window

In [81]:
score_tol = meta.get("score_tolerance", 1e-6)
rows = []

for key, case in cases.items():
    raw_window = rebuild_raw_window(case)

    # Data integrity check: did the source .npy file change under us?
    actual_hash = hashlib.sha256(raw_window.tobytes()).hexdigest()
    hash_ok = actual_hash == case["raw_window_sha256"]

    X = preprocess_live_window(case["ch_id"], raw_window, manager)
    result = manager.predict(case["ch_id"], X)

    actual_label = int(result["label"][0])
    actual_score = float(result["score"][0])

    label_ok = actual_label == case["expected_label"]
    score_diff = abs(actual_score - case["expected_score"])
    score_ok = score_diff <= score_tol

    rows.append({
        "case": key,
        "ch_id": case["ch_id"],
        "input_hash_ok": hash_ok,
        "expected_label": case["expected_label"],
        "actual_label": actual_label,
        "label_ok": label_ok,
        "expected_score": round(case["expected_score"], 6),
        "actual_score": round(actual_score, 6),
        "score_diff": score_diff,
        "score_ok": score_ok,
        "PASS": hash_ok and label_ok and score_ok,
    })

import pandas as pd
results_df = pd.DataFrame(rows)
pd.set_option("display.width", 140)
results_df

,case,ch_id,input_hash_ok,expected_label,actual_label,label_ok,expected_score,actual_score,score_diff,score_ok,PASS
0,P-1_offset0,P-1,True,1,1,True,0.151138,0.151138,5.551115e-17,True,True
1,P-1_offset2835,P-1,True,1,1,True,0.102217,0.102217,0.000000e+00,True,True
2,P-1_offset5670,P-1,True,1,1,True,0.137944,0.137944,0.000000e+00,True,True
3,E-1_offset0,E-1,True,1,1,True,0.188637,0.188637,0.000000e+00,True,True
4,E-1_offset2838,E-1,True,1,1,True,0.159994,0.159994,0.000000e+00,True,True
5,E-1_offset5677,E-1,True,-1,-1,True,-0.162447,-0.162447,0.000000e+00,True,True
6,A-1_offset0,A-1,True,1,1,True,0.065707,0.065707,0.000000e+00,True,True
7,A-1_offset2880,A-1,True,1,1,True,0.108016,0.108016,0.000000e+00,True,True
8,A-1_offset5760,A-1,True,1,1,True,0.039586,0.039586,0.000000e+00,True,True
9,T-1_offset0,T-1,True,-1,-1,True,-0.237063,-0.237063,0.000000e+00,True,True


In [82]:
n_total = len(results_df)
n_pass = int(results_df["PASS"].sum())
n_fail = n_total - n_pass

print(f"{n_pass}/{n_total} regression cases passed.")

if n_fail:
    print(f"\n{n_fail} FAILING case(s):")
    display_cols = ["case", "input_hash_ok", "expected_label", "actual_label",
                     "expected_score", "actual_score", "score_diff"]
    print(results_df[~results_df["PASS"]][display_cols].to_string(index=False))
else:
    print("No prediction drift detected against the frozen golden values.")

assert n_fail == 0, (
    f"{n_fail} regression case(s) failed -- see table above. "
    f"If this follows an intentional retrain, regenerate tests/fixtures/regression_fixtures.json "
    f"(see the last cell) instead of treating this as a bug."
)

24/24 regression cases passed.
No prediction drift detected against the frozen golden values.


In [83]:
manager_a = MultiChannelIsolationForest.load(MODEL_PATH)
manager_b = MultiChannelIsolationForest.load(MODEL_PATH)

mismatches = []
for key, case in list(cases.items())[:12]:  # a representative subset is enough here
    raw_window = rebuild_raw_window(case)
    Xa = preprocess_live_window(case["ch_id"], raw_window, manager_a)
    Xb = preprocess_live_window(case["ch_id"], raw_window, manager_b)
    ra = manager_a.predict(case["ch_id"], Xa)
    rb = manager_b.predict(case["ch_id"], Xb)
    same_label = int(ra["label"][0]) == int(rb["label"][0])
    same_score = float(ra["score"][0]) == float(rb["score"][0])  # bit-exact expected
    if not (same_label and same_score):
        mismatches.append(key)

print(f"Checked {min(12, len(cases))} cases across two independently loaded model instances.")
print("PASS -- identical predictions from both loads." if not mismatches
      else f"FAIL -- mismatches in: {mismatches}")
assert not mismatches

Checked 12 cases across two independently loaded model instances.
PASS -- identical predictions from both loads.
